# ЛР2. Пункт 10. Значимость регрессионных коэффициентов

Для каждого коэффициента выполняем индивидуальный двусторонний t-тест
при уровне значимости 0,05 по слайдам 8–11 лекции 3.

## Воспроизведение модели ЛР1

Из `all_data_processed.csv` берём 1365 строк с меткой `split=train`. Оцениваем МНК по этим строкам. Совпадение прогнозов проверяем по `predictions.csv`.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display


data_dir = Path(".") if Path("predictions.csv").is_file() else Path("lab2")

data = pd.read_csv(data_dir / "all_data_processed.csv")
training = data.loc[data["split"].eq("train")]
feature_names = [
    "log1p_cores", "base_clock_ghz", "log1p_tdp", "log1p_l3",
    "age_centered", "manufacturer_Intel", "manufacturer_Other", "category_Laptop",
]
X = np.column_stack([np.ones(len(training)), training[feature_names]])
y = training["ln_cpu_mark"].to_numpy()
coefficients = np.linalg.lstsq(X, y, rcond=None)[0]
y_hat = X @ coefficients

saved_predictions = pd.read_csv(data_dir / "predictions.csv")["y_hat"]
np.testing.assert_allclose(y_hat, saved_predictions, rtol=0, atol=1e-10)

In [2]:
extra_sample = pd.read_csv(data_dir / "cpu_extra_sample.csv")
y_extra = np.log(extra_sample["cpu_mark"].to_numpy())
df_0 = len(y_extra) - 1
S0_sq = y_extra.var(ddof=1)

# Сопоставляем записи дополнительной группы по всем её сохранённым полям.
shared_columns = extra_sample.columns.tolist()
development = training[["raw_" + column for column in shared_columns]].set_axis(shared_columns, axis=1)
extra_keys = pd.MultiIndex.from_frame(extra_sample[shared_columns])
development_keys = pd.MultiIndex.from_frame(development[shared_columns])
overlap = extra_keys.isin(development_keys).sum()

print(f"Дополнительная группа: {len(extra_sample)} CPU, ν₀ = {df_0}")
print(f"S₀² = {S0_sq:.6f}")
print(f"Совпадений с обучающей выборкой: {overlap} из {len(extra_sample)}")

Дополнительная группа: 20 CPU, ν₀ = 19
S₀² = 0.080688
Совпадений с обучающей выборкой: 14 из 20


## Стандартные ошибки и t-тесты

Для каждого $j$ проверяем $H_0:\beta_j=0$ против $H_1:\beta_j\ne 0$.
Из ковариационной матрицы $S_0^2C$ получаем
$\mathrm{SE}(b_j)=\sqrt{S_0^2 C_{jj}}$ и
$t_j=b_j/\mathrm{SE}(b_j)$.
Число степеней свободы берём из оценки $S_0^2$.
Гипотеза отклоняется при $|t_j|>t_{1-\alpha/2,\nu_0}$.

In [3]:
from scipy.stats import t

alpha = 0.05
terms = ["const", *feature_names]
C = pd.read_csv(data_dir / "error_matrix.csv")
# Строки CSV записаны в порядке его столбцов.
C.index = C.columns
C = C.loc[terms, terms].to_numpy()
np.testing.assert_allclose(C @ (X.T @ X), np.eye(len(terms)), rtol=0, atol=1e-10)

covariance = S0_sq * C
standard_errors = np.sqrt(np.diag(covariance))
t_values = coefficients / standard_errors
t_critical = t.isf(alpha / 2, df_0)
p_values = 2 * t.sf(np.abs(t_values), df_0)
significant = p_values < alpha

labels = [
    "Свободный член", "Число ядер", "Базовая частота", "TDP", "Кэш L3",
    "Возраст", "Intel относительно AMD", "Other относительно AMD",
    "Laptop относительно Desktop",
]
results = pd.DataFrame({
    "Фактор": labels,
    "Коэффициент": coefficients,
    "Стандартная ошибка": standard_errors,
    "t": t_values,
    "p-value": p_values,
    "Значим при α = 0,05": significant,
}, index=pd.Index(terms, name="Регрессор"))

print(f"ν₀ = {df_0}; критическое |t| = {t_critical:.4f}")
print(results.to_string(formatters={
    "Коэффициент": "{:.6f}".format,
    "Стандартная ошибка": "{:.6f}".format,
    "t": "{:.3f}".format,
    "p-value": "{:.3g}".format,
}))

ν₀ = 19; критическое |t| = 2.0930
                                         Фактор Коэффициент Стандартная ошибка       t  p-value  Значим при α = 0,05
Регрессор                                                                                                           
const                            Свободный член    7.172794           0.007688 932.932  9.5e-46                 True
log1p_cores                          Число ядер    0.483894           0.013633  35.494 7.82e-19                 True
base_clock_ghz                  Базовая частота    0.346360           0.013190  26.259 2.15e-16                 True
log1p_tdp                                   TDP    0.108749           0.014152   7.684 3.03e-07                 True
log1p_l3                                 Кэш L3    0.203161           0.014363  14.145 1.54e-11                 True
age_centered                            Возраст   -0.319625           0.010660 -29.985 1.83e-17                 True
manufacturer_Intel       Intel

In [4]:
print("Направления связей с ln(CPU Mark) при фиксированных остальных факторах:")
for label, estimate, is_significant in zip(labels[1:], coefficients[1:], significant[1:]):
    if is_significant:
        direction = "положительная" if estimate > 0 else "отрицательная"
        print(f"{label} — {direction} связь.")
    else:
        print(f"{label} — недостаточно оснований считать коэффициент отличным от нуля.")

Направления связей с ln(CPU Mark) при фиксированных остальных факторах:
Число ядер — положительная связь.
Базовая частота — положительная связь.
TDP — положительная связь.
Кэш L3 — положительная связь.
Возраст — отрицательная связь.
Intel относительно AMD — положительная связь.
Other относительно AMD — отрицательная связь.
Laptop относительно Desktop — положительная связь.


Выводы о факторах предварительные. Оценка $S_0^2$ приближённая,
а часть записей дополнительной группы входит в обучающую выборку.
По лекции 3 при интерпретации индивидуальных тестов учитываем результаты
проверки адекватности в пункте 9 и однородности дисперсий в пункте 11.
Для категориальных факторов направление относится к сравнению с указанной
базовой категорией. Связи описывают наблюдательные данные.